### Middleware

Middleware provides a way to more tightly control what happens inside the agent.Middleware is useful for the following:
    
    1.Tracking agent behaviour with logging, analytics and debugging
    2.Transforming prompts, tool selection, and output formatting.
    3.Adding retries, fallbacks and early termination logic.
    4.Applying rate limits, guardrails and PII detection

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["MISTRAL_API_KEY"] = os.getenv("MISTRAL_API_KEY")

### Summarization Middleware

Automatically summarized conversation history when approaching token limits, preserving recent messaging while compressing older context.Summarization is useful for the following.

- Long running concersations that exceeded context windows.
- Multi-turn dialogues with extensive history.
- Applications where preseving full conversation context matters.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage


# Message based summarization 
agent = create_agent(
    model="mistral-medium-3-5",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="mistral-medium-3-5",
            trigger=("messages", 10),
            keep=("messages",4)
        )
    ]

)

config = {"configurable":{"thread_id":"test-1"}}

questions = [
    "What is 1+1 ?",
    "Whats is 2*2 ?",
    "What is 3+4 ?",
    "What is 1+3 ?",
    "What is 5-2 ?",
    "What is 6+2 ?"
]
for q in questions:
     response = agent.invoke({"messages":[HumanMessage(content=q)]}, config)
     print(f"Message : {response}")
     print(f"Message:{len(response['messages'])}")

Message : {'messages': [HumanMessage(content='What is 1+1 ?', additional_kwargs={}, response_metadata={}, id='46b42136-fa6d-4209-ac8a-cba6a491bb58'), AIMessage(content='The sum of 1 + 1 is **2**.', additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 22, 'total_tokens': 35, 'completion_tokens': 13, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-3-5', 'model': 'mistral-medium-3-5', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--019f03f6-cc86-7110-87fe-9593dfdc4a26-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 22, 'output_tokens': 13, 'total_tokens': 35})]}
Message:2
Message : {'messages': [HumanMessage(content='What is 1+1 ?', additional_kwargs={}, response_metadata={}, id='46b42136-fa6d-4209-ac8a-cba6a491bb58'), AIMessage(content='The sum of 1 + 1 is **2**.', additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 22, 'total_tokens': 35, 'completion_tokens': 13, 'prom

### token size

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotel(city:str) -> str:
    """Search hotels - returns long response to use  more tokens"""
    return f"""Hotels in {city}:
    1 . Grand Hote- 5 star, $350/night, spa,pool,gym
    2. City Inn - 4 star,$180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent = create_agent(
    model="mistral-medium-3-5",
    tools=[search_hotel],
    checkpointer=InMemorySaver(),
    middleware= [
        SummarizationMiddleware(
            model="mistral-medium-3-5",
            trigger=("tokens", 450),
            keep=("tokens", 200)
        )
    ]
)

config = {"configurable":{"thread_id":"test-1"}}


#token counter 
def count_tokens(messages):
    total_chars = sum(len(str(m.content))for m in messages)
    return total_chars // 4 # 4 chars = 1 token

In [9]:
cities = ["Paris", "London", "NewYork", "Tokyo", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response["messages"])}messages")
    print(f"{(response["messages"])}")

Paris: ~134 tokens, 4messages
[HumanMessage(content='find hotels in Paris', additional_kwargs={}, response_metadata={}, id='a1d0bf12-523f-4b8e-89f4-bdad8a916e32'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'fjODQbi1Y', 'type': 'function', 'function': {'name': 'search_hotel', 'arguments': '{"city": "Paris"}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 84, 'total_tokens': 96, 'completion_tokens': 12, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-3-5', 'model': 'mistral-medium-3-5', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019f0406-c989-79b3-80f0-e9fbce226edb-0', tool_calls=[{'name': 'search_hotel', 'args': {'city': 'Paris'}, 'id': 'fjODQbi1Y', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 84, 'output_tokens': 12, 'total_tokens': 96}), ToolMessage(content='Hotels in Paris:\n    1 . Grand Hote- 5 star, $350/night, spa,pool,gym\n    2. City Inn - 4 

### Fraction

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

@tool

def search_hotel(city: str) -> str:
    """Search hotels."""
    return f"Hotels in the {city}: Grang hotel $350, City Inn $250,Budget Stay $75"

# Low fraction for testing

agent = create_agent(
    model="mistral-medium-3-5",
    tools=[search_hotel],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="mistralai:mistral-medium-3-5",
            trigger=("fraction", 0.005),
            keep=("fraction", 0.0025),
        )
    ]
)

config = {"configurable":{"thread_id":"test-1"}}

#Token Counter

def count_tokens(messages):
    return sum(len(str(m.content))for m in messages)

# Test

cities =["Paris", "London", "NewYork", "Tokyo", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"find Hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    fraction= tokens/128000
    print(f"{city}:~{tokens} tokens ({fraction:.4%}), {len(response["messages"])}msgs")
    print(response["messages"])

ValueError: Model profile information is required to use fractional token limits, and is unavailable for the specified model. Please use absolute token counts instead, or pass `

ChatModel(..., profile={"max_input_tokens": ...})`.

with a desired integer value of the model's maximum input tokens.

### Human in the Loop

In [33]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str)-> str:
    """Mock function to read emails by it Ids"""
    return f"Email content for id :{email_id}"

def send_email_tool(email_id: str)-> str:
    """Mock function to send an email"""
    return f"Email sent to {recipent} with subject '{subject}'"

In [34]:
agent = create_agent(
    model="mistral-medium-3-5",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
       HumanInTheLoopMiddleware(
           interrupt_on={
               "send_email_tool":{
                   "allowed_decisions": ["approved", "edit", "reject"],
               },
               "read_email_tool": False,
           }
       )
    ]
)

In [35]:
config = {"configurable":{"thread_id":"test-approve"}}

result = agent.invoke(
    {"messages":[HumanMessage(content="send email to johan@test.com with subject'Hello' and body 'How are you?'")]},
    config=config
)

In [36]:
result

{'messages': [HumanMessage(content="send email to johan@test.com with subject'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='a5d1a540-f1a3-4cf4-8087-b1ee1d98edb9'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ob3FWcrQy', 'type': 'function', 'function': {'name': 'send_email_tool', 'arguments': '{"email_id": "johan@test.com_Hello_How are you?"}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 161, 'total_tokens': 186, 'completion_tokens': 25, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-3-5', 'model': 'mistral-medium-3-5', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019f04da-e38e-71a0-869f-d2a361ddcd28-0', tool_calls=[{'name': 'send_email_tool', 'args': {'email_id': 'johan@test.com_Hello_How are you?'}, 'id': 'ob3FWcrQy', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 161, 'output_tokens': 25, 'total_tokens': 186})],
 

In [ ]:
from langgraph.types import Command
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type" : "approved"}
                ]
            }
        ),
        config=config
    )
    print(f"✅ Result:{result['messages'][-1].content}")

⏸️ Paused! Approving...


ValueError: Unexpected human decision: {'type': 'approve'}. Decision type 'approve' is not allowed for tool 'send_email_tool'. Expected one of ['approved', 'edit', 'reject'] based on the tool's configuration.